In [229]:
model = "llama3.2:1B"

In [ ]:
import sys
!{sys.executable} -m pip install wikipedia -q

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [230]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = "llama3.2:1B"

In [231]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="llama3.2:1B")


In [232]:

from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection(name="my_collection")


# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(collection_name="my_collection", embedding_function=embedding_model, client=client)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
import wikipedia

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)

# ============= HINTERLEGTE DATENQUELLEN =============
DATA_SOURCES = [
    {
        "type": "wikipedia",
        "query": "Maschinelles Lernen",
        "lang": "de",
        "url": "https://de.wikipedia.org/wiki/Maschinelles_Lernen"
    },
    {
        "type": "wikipedia",
        "query": "Künstliche Intelligenz",
        "lang": "de",
        "url": "https://de.wikipedia.org/wiki/Künstliche_Intelligenz"
    },
]

# ============= LADE ALLE DOKUMENTE =============
all_docs = []

for source in DATA_SOURCES:
    try:
        if source["type"] == "wikipedia":
            wikipedia.set_lang(source["lang"])
            page = wikipedia.page(source["query"])
            doc = Document(
                page_content=page.content,
                metadata={
                    "source": page.url,
                    "title": page.title,
                    "language": source["lang"],
                    "source_url": source["url"]
                }
            )
            all_docs.append(doc)
            print(f"✓ Geladen: {source['query']} (1 Dokument)")
    except Exception as e:
        print(f"✗ Fehler bei {source['query']}: {str(e)}")

# Fallback mit Mock-Dokumenten wenn nichts geladen wurde
if not all_docs:
    print("\n⚠ Nutze Mock-Dokumente als Fallback\n")
    all_docs = [
        Document(
            page_content="""Maschinelles Lernen ist ein Oberbegriff für die künstliche Erzeugung von Wissen aus Erfahrung: Ein lernender Agent verbessert seine Fähigkeiten, indem er eine Aufgabe wiederholt löst und dabei aus fehlerhaften Versuchen lernt.""",
            metadata={"source": "Wikipedia ML", "source_url": "https://de.wikipedia.org/wiki/Maschinelles_Lernen"}
        ),
        Document(
            page_content="""Künstliche Intelligenz (KI, englisch artificial intelligence, AI) ist ein Teilgebiet der Informatik, das sich mit der Automatisierung intelligenten Verhaltens befasst.""",
            metadata={"source": "Wikipedia KI", "source_url": "https://de.wikipedia.org/wiki/Künstliche_Intelligenz"}
        )
    ]

# Collection leeren um doppelte/alte Einträge zu vermeiden
client.delete_collection(name="my_collection")
fresh_collection = client.get_or_create_collection(name="my_collection")
vector_db_from_client = Chroma(
    collection_name="my_collection",
    embedding_function=embedding_model,
    client=client
)

# Speichere alle Dokumente in Chroma Vector DB
vector_db_from_client.add_documents(all_docs)
print(f"\n✓ Insgesamt {len(all_docs)} Dokumente in Chroma gespeichert\n")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = prompt | ChatOllama(model=model) | StrOutputParser()

result = chain.invoke({"docs": format_docs(all_docs[:3])})
print(result)

In [281]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query, k=5)

print(docs)

[Document(id='ml_doc_1', metadata={}, page_content='Machine learning is a branch of artificial intelligence that focuses on \n    building applications that learn from data and improve their performance over time without \n    being explicitly programmed. It involves algorithms and statistical models that enable \n    computers to learn from and make decisions or predictions based on data.\n\n    Key types include supervised learning (with labeled data), unsupervised learning (pattern discovery),\n    and reinforcement learning (learning through interaction). Common applications include image recognition,\n    natural language processing, recommendation systems, and predictive analytics.'), Document(id='1cb910c3-829d-44be-9778-52b0dbcfa309', metadata={'source': 'Wikipedia ML', 'url': 'https://de.wikipedia.org/wiki/Maschinelles_Lernen'}, page_content='Maschinelles Lernen ist ein Oberbegriff für die künstliche Erzeugung von Wissen aus Erfahrung: Ein lernender Agent verbessert seine Fähig

In [282]:
chain.invoke({"docs": format_docs(docs)})

'Der Artikel "Maschinelles Lernen" beschreibt die Grundlagen und Anwendungen des maschinellen Lernens. Maschinelles Lernen ist eine disziplin, die von der Kognitionswissenschaft aufgebaut ist und sich in der Forschungsmethoden zur Beantwortung komplexer Probleme wie Bildanalyse, Textanalysen und Datenverarbeitung unterscheidet.\n\nDie Geschichte des maschinellen Lernens reicht zurück ins 20. Jahrhundert, als die erste Maschine, den Prinzipien von Alan Turing nachgegangen war, entwickelt wurde. Im Laufe der Zeit entwickelte sich das maschinelle Lernen weiter in verschiedenen Bereichen wie Künstlicher Intelligenz, Datenverarbeitung und künstlichen Begriffen.\n\nEinige Schlüsselaspekte des maschinellen Lernens sind:\n\n* **Algorithmen**: Maschinelles Lernen basiert auf Algorithmen, die spezielle Aufgaben erfüllen können. Im Fall von Klassifikationen werden Algorithmen entwickelt, um zwischen verschiedenen Klassen zu unterscheiden.\n* **Neuronale Netze**: Eine grundlegende Struktur in der 

In [283]:
# Simple stream the chain output
for chunk in chain.stream({"docs": format_docs(docs)}):
    print(chunk, end="", flush=True)

Maschinelles Lernen ist ein Fachgebiet der Informatik, das sich mit der Entwicklung und Anwendung von Algorithmen und Künstlichen Intelligenz (KI) für verschiedene Zwecke konzentriert. Es umfasst die Analyse und Modellierung von Daten, zur Gewinnung von Entscheidungen oder zum Einhalt von Regeln.

Einige der wichtigsten Aspekte des Maschinelles Lernens sind:

* **Klassifizierung**: Die Unterteilung von Daten in verschiedene Klassen oder Begriffe.
* **Regressionsanalyse**: Die Verwendung von Algorithmen, um die Beziehung zwischen einer Variablen und einem Ergebnis zu modellieren.
* **Unterstützende Schätze (Supervised Learning)**: Die Verwendung von Algorithmen zur Interpretation von Daten und der Vorhersage des Ergebnisses auf der Grundlage dieser Daten.
* **Neuronale Netze (Neural Networks)**: Eine Art von maschinellen Lernmodell, bei dem sich die Netze mit Hilfe von Neuronen verbindet und so durch Lernen automatisch komplexe Verbindungen entwickeln können.

Einige der wichtigsten Anw

In [284]:
# More complex async event streaming
async for event in chain.astream_events({"docs": format_docs(docs)}, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Es gibt mehrere Arten von maschinellen Lernmodellen. Einige der wichtigsten Arten sind:

1. **Klassifikationsmodelle**: Diese Modelle werden verwendet, um zwischen verschiedenen Kategorien zu unterscheiden (z.B. Bildklassifikation). Beispiele hierfür sind die CNNs für das Bildklassifikationsproblem.

2. **Regressionsmodelle**: Diese Modelle werden verwendet, um einen Ausdruck als Antwort auf eine Anfrage zu liefern. Ein Beispiel hierfür ist der Schatzmodell für das Angebot in einer Online-Plattform.

3. **Anfangspunktsmodelle**: Diese Modelle sind in den frühen Stadien des Lernprozesses sehr hilfreich, um die Bedingungen und Parameter eines Modells zu identifizieren.

4. **Deep-Learning-Modelle**: Diese Modelle haben eine hohe Anzahl von Schichten (oft zwischen 10-100) und verwenden viele verschiedene Algorithmen zur Verarbeitung von Daten.

5. **Neural Networks**: Diese Modelle basieren auf einer Reihe von Schichten mit einfachen oder komplexen Verbindungen und werden in vielen Bereic

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [285]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever(search_kwargs={"k": 5})

# ADD HERE YOUR CODE
qa_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | ChatOllama(model=model)
    | StrOutputParser()
)

In [286]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000020DBF3356D0>, search_kwargs={'k': 5})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"), additional_kwargs={})])
| ChatOllama(model='llama3.2:1B')
| StrOutputParser()

In [287]:
# Prüfe explizit, welche Dokumente eingebunden sind
print("=== Überprüfung der eingebundenen Daten ===\n")

# 1. Anzahl der Dokumente in der Collection
count = collection.count()
print(f"Anzahl Dokumente in Chroma: {count}\n")

# 2. Direkte Abfrage des Retrievers
retrieved_docs = retriever.invoke("Maschinelles Lernen")
print(f"Anzahl abgerufener Dokumente: {len(retrieved_docs)}\n")

# 3. Quellen und Inhalte der Dokumente anzeigen
print("=== Inhalte der abgerufenen Dokumente ===\n")
for i, doc in enumerate(retrieved_docs):
    print(f"Dokument {i+1}:")
    print(f"Quelle: {doc.metadata}")
    print(f"Inhalt (erste 200 Zeichen): {doc.page_content[:200]}...")
    print("-" * 80 + "\n")

# Jetzt die Frage für die nächsten Zellen
question = "Was ist der erste Satz, des Wikipedia-Artikels https://de.wikipedia.org/wiki/Maschinelles_Lernen"

# ADD HERE YOUR CODE
result = qa_rag_chain.invoke(question)
print("\n=== RAG Chain Antwort ===\n")
print(result)

=== Überprüfung der eingebundenen Daten ===

Anzahl Dokumente in Chroma: 5

Anzahl abgerufener Dokumente: 5

=== Inhalte der abgerufenen Dokumente ===

Dokument 1:
Quelle: {'url': 'https://de.wikipedia.org/wiki/Maschinelles_Lernen', 'source': 'Wikipedia ML'}
Inhalt (erste 200 Zeichen): Maschinelles Lernen ist ein Oberbegriff für die künstliche Erzeugung von Wissen aus Erfahrung: Ein lernender Agent verbessert seine Fähigkeiten, indem er eine Aufgabe wiederholt löst und dabei aus feh...
--------------------------------------------------------------------------------

Dokument 2:
Quelle: {}
Inhalt (erste 200 Zeichen): Machine learning is a branch of artificial intelligence that focuses on 
    building applications that learn from data and improve their performance over time without 
    being explicitly programmed...
--------------------------------------------------------------------------------

Dokument 3:
Quelle: {'title': 'Maschinelles Lernen – Wikipedia', 'source': 'https://de.wik

In [288]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Der erste Satz des Wikipedia-Artikels Maschinelles Lernen lautet: "Maschinelles Lernen ist eine Kombination aus Ingenieurwissenschaften und Wirtschaftswissenschaften, die den Einsatz von Computerprogrammen und künstlichen Intelligenzen zur Analyse und Modellierung von Daten verwendet."

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [289]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [290]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [291]:
qa_chain = ConversationalRetrievalChain.from_llm(
    ChatOllama(model=model), retriever=retriever, memory=memory, verbose=False
)

In [292]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning ist eine Art von Machine-Learning, bei der ein Model ermittelt werden soll, was einem Trainingsdaten basierend auf einer bestimmten Beziehung zwischen Variablen und Target variable passiert. Das Ziel des Trainingstages ist es, das Model so zu trainieren, dass es die richtige Antwort an einen bestimmten Test-Beispiel geben kann.

Ein Beispiel:

Stellen Sie sich vor, Sie möchten ein automatisches Fahrzeug steuern. Die Beziehung zwischen Variablen könnte wie folgt lauten:

* Unbequemlichkeit der Steuerung (Variable X) -> Ausfuhrgeräte des Fahrzeugs (Target variable)
* Geschwindigkeit (Variable Y) = Ausfuhrgeräte des Fahrzeugs
* Zeitpunkt der Steuerung (Variable Z) = Zeitpunkt des Fahrradschritts

Ein supervised-lernendes Modell wird erzeugt, das die Beziehung zwischen Unbequemlichkeit, Geschwindigkeit und Zeitpunkt der Steuerung anhand einer großen Menge von Trainingsdaten (Trenndaten) festlegt. Die Trenndaten umfassen alle möglichen Paare von Variablen und Target vari

In [293]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Which machine learning techniques are commonly employed in the field of autonomous vehicles, such as supervised and unsupervised learning methods?In the field of autonomous vehicles, several machine learning techniques are commonly employed to enable self-driving cars. Here are some of the most common ones:

**Supervised Learning Methods:**

1. **Object Detection**: This involves identifying and classifying objects in the vehicle's environment, such as pedestrians, cars, road signs, etc.
2. **Image Classification**: Object detection is often followed by image classification to identify specific objects or scenes within an image.
3. **Object Tracking**: This technique enables the vehicle to track specific objects over time, allowing it to predict their location and movement.

**Unsupervised Learning Methods:**

1. **Clustering Analysis**: Clustering helps to group similar objects together, enabling the vehicle to identify patterns in the environment.
2. **Density Estimation**: Density e